## 评估

In [ ]:
import csv

def parse_score(value):
    """把score转成0/1或浮点数：False/0->0, True/1->1，其它尝试float。"""
    if value is None:
        return None
    v = str(value).strip()

    if v.lower() == "true":
        return 1.0
    if v.lower() == "false":
        return 0.0
    if v == "1":
        return 1.0
    if v == "0":
        return 0.0

    try:
        return float(v)
    except ValueError:
        return None


def compute_scores(csv_path):
    total_sum = 0.0
    total_count = 0

    type_stats = {}  

    with open(csv_path, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            raw_score = row.get("score")
            q_type = row.get("question_type", "").strip()

            score = parse_score(raw_score)
            if score is None:
                continue

            total_sum += score
            total_count += 1

            if q_type not in type_stats:
                type_stats[q_type] = {"sum": 0.0, "count": 0}
            type_stats[q_type]["sum"] += score
            type_stats[q_type]["count"] += 1

    if total_count > 0:
        overall_avg = total_sum / total_count
    else:
        overall_avg = 0.0

    print(f"整体平均分: {overall_avg:.4f}  (样本数: {total_count})")

    print("\n按 question_type 分组的平均分：")
    for q_type, stats in type_stats.items():
        if stats["count"] == 0:
            avg = 0.0
        else:
            avg = stats["sum"] / stats["count"]
        print(f"- {q_type}: {avg:.4f}  (样本数: {stats['count']})")


if __name__ == "__main__":
    csv_file_path = "results/Memrewriter/eval_results_32k.csv"
    compute_scores(csv_file_path)


In [ ]:
# 计算属性树的平均长度，英文
import json

def average_value_length(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)   # 假设顶层是一个 dict

    total_len = 0
    count = 0

    for key, value in data.items():
        if isinstance(value, dict):
            s = json.dumps(value, ensure_ascii=False)
        else:
            s = str(value)  # 兜底：不是 dict 也转成字符串

        total_len += len(s.split())
        count += 1

    avg_len = total_len / count if count > 0 else 0
    print(f"键的数量: {count}")
    print(f"所有值转成字符串后的平均长度: {avg_len}")

# 把这里换成你的 JSON 文件路径
json_file = "data/shared_contexts_32k/all_final_tree.json"
average_value_length(json_file)

In [ ]:
# 调用大模型
from src.llm import Agent
agent = Agent("You are a helpful assistant")
response, success = agent.run("你好")
if success:
    print(response)
else:
    print(response)

## 测试mongoDB

In [ ]:
from pymongo import MongoClient

# === 1. Mongo 连接 ===
# 按你实际环境改连接字符串 / 库名 / 集合名
client = MongoClient("mongodb://ip:port")
db = client["persona_mem"]
tree_col = db["attribute_trees"]  

In [ ]:
# 只需要创建一次

from datetime import datetime
from zoneinfo import ZoneInfo
tree_col.update_one(
        {"user_id": '123'},
        {
            "$setOnInsert": {
                "tree": {
                        "1_Biological_Characteristics": {
                            "Physical_Appearance": {
                                "Body_Build": {
                                    "Height": "",
                                    "Weight": "",
                                    "Body_Habitus": ""
                                },
                                "Facial_Features": {
                                    "Face_Shape": "",
                                    "Eyes": "",
                                    "Nose": "",
                                    "Mouth": "",
                                    "Ears": "",
                                    "Eyebrows": "",
                                    "Teeth": ""
                                }
                            }
                        }
                    }, 
                "created_at": datetime.now(ZoneInfo("Asia/Shanghai")),
            }
        },
        upsert=True,
    )

In [ ]:
mongo_path = "tree.1_Biological_Characteristics.Physical_Appearance.Facial_Features.Ears"
tree_col.update_one(
    {"user_id": '123'},
    {
        "$set": {
            mongo_path: '2',
            "updated_at": datetime.now(ZoneInfo("Asia/Shanghai")),
        }
    }
)

In [ ]:
import json
doc = tree_col.find_one(
        {"user_id": '123'},
        # {"_id": 0,"tree": 1 }      # 只要 tree 字段
    )
print(doc)
# print(json.dumps(doc, ensure_ascii=False, indent=2))

In [ ]:
mongo_path = f"tree.1_Biological_Characteristics.Facial_Features"
tree_col.update_one(
    {"user_id": '123'},
    {
        "$unset": {mongo_path: ""},
        "$set": {"updated_at": datetime.now(ZoneInfo("Asia/Shanghai"))},
    }
)